In [0]:
import sys

project_root = dbutils.secrets.get(scope="project_scope", key="project_root")
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.silver_tools import silverLayer

In [0]:
%sql
create schema if not exists project_cat.silver_layer;
create schema if not exists project_cat.quarantine_layer;

In [0]:
# dim_customers table

# transform raw data from bronze layer

s = silverLayer.get_config("dim_customers")
df = s.get_bronze_data()
key_null_df = s.get_key_null(df)
dup_data = s.get_dup_data(df, key_null_df)
bad_data = s.get_bad_data(key_null_df, dup_data)
cleaned_data = s.get_cleaned_data(df, bad_data)

# load cleaned data into silver layer
s.load_into_silver(cleaned_data)
s.load_bad_data(bad_data)

In [0]:
# dim_products table

# transform
s = silverLayer.get_config("dim_products")
df = s.get_bronze_data()
key_null_df = s.get_key_null(df)
dup_data = s.get_dup_data(df, key_null_df)
bad_data = s.get_bad_data(key_null_df, dup_data)
cleaned_data = s.get_cleaned_data(df, bad_data)

# load
s.load_into_silver(cleaned_data)
s.load_bad_data(bad_data)

In [0]:
# fact_saless table need to be dropped some duplicate columns

s = silverLayer.get_config("fact_sales")
bronze_df = s.get_bronze_data()

# drop columns that duplicate with dim_customers and dim_products
df = s.drop_dup_columns(bronze_df, "dim_customers", "dim_products")

# transform
key_null_df = s.get_key_null(df)
dup_data = s.get_dup_data(df, key_null_df)
bad_data = s.get_bad_data(key_null_df, dup_data)
cleaned_data = s.get_cleaned_data(df, bad_data)

#load
s.load_into_silver(cleaned_data)
s.load_bad_data(bad_data)

In [0]:
# fact_orders table need to be dropped some duplicate columns and concat a date column

s = silverLayer.get_config("fact_orders")
bronze_df = s.get_bronze_data()

#concat date column
concat_df = (bronze_df.withColumn("order_date_time", to_timestamp(concat_ws(" ", col("order_date"), col("order_time"))))
                        .drop("order_date", "order_time"))

# drop columns that duplicate with dim_customers and dim_products
df = s.drop_dup_columns(concat_df , "dim_customers", "dim_products")

# transform
key_null_df = s.get_key_null(df)
dup_data = s.get_dup_data(df, key_null_df)
bad_data = s.get_bad_data(key_null_df, dup_data)
cleaned_data = s.get_cleaned_data(df, bad_data)

# load
s.load_into_silver(cleaned_data)
s.load_bad_data(bad_data)